In [1]:
import numpy as np
import math
from tqdm import tqdm

In [2]:
def count_adjacencies(point, points):
    count = 0
    #If for any dimension d , the absolute difference is larger than 1, not adjacent
    for other in points:
        is_adjacent = True
        for dim in range(len(point)):
            diff = abs(point[dim] - other[dim])
            if diff > 1 and diff < 6:
                is_adjacent = False
                break
        if is_adjacent:
            count += 1    
    return count

def return_adjacencies(points):
    adjacency_counts = []
    for point in points:
        count = count_adjacencies(point, points)
        adjacency_counts.append(count-1)  # Subtract 1 to not count itself
    return adjacency_counts

def create_independent_points(points):
    adjacency_counts = return_adjacencies(points)
    while (max(adjacency_counts) > 0):
        max_index = adjacency_counts.index(max(adjacency_counts))
        points = np.delete(points, max_index, axis=0)
        adjacency_counts = return_adjacencies(points)
    return points

def generate_balanced_points(n, k, d, max_attempts):
    """
    Generate k DISTINCT points in d dimensions where each dimension uses 
    values 0-(n-1) equally. Uses a hybrid approach: random generation with
    fallback to deterministic filling.
    
    Parameters:
    k: number of points (must be divisible by n)
    d: number of dimensions
    max_attempts: maximum number of attempts to generate distinct points
    
    Returns:
    numpy array of shape (k, d) with all distinct points
    """
    if k % n != 0:
        raise ValueError(f"k must be divisible by {n}. Got k={k}")
    
    if k > n**d:
        raise ValueError(f"Cannot generate {k} distinct points in {d} dimensions with values 0-n-1. Maximum possible: {n**d}")
    
    # Number of times each value appears
    repeats = k // n
    
    # Try random generation for a fraction of max_attempts
    random_attempts = max(1, max_attempts // 2)
    
    for attempt in tqdm(range(random_attempts), desc="Random generation"):
        # Create points
        points = np.zeros((k, d), dtype=int)
        
        for dim in range(d):
            # Create balanced array: each value 0-(n-1) appears 'repeats' times
            values = np.repeat(np.arange(n), repeats)
            # Shuffle to randomize positions
            np.random.shuffle(values)
            points[:, dim] = values
        
        # Check if all points are distinct
        unique_points = np.unique(points, axis=0)
        if len(unique_points) == k:
            print(f"✓ Generated {k} distinct points after {attempt+1} random attempts.")
            return points
    
    # Fallback: hybrid approach - start random, then fill deterministically
    print(f"Random generation didn't succeed. Switching to hybrid approach...")
    
    # Use a set to track points for fast duplicate checking
    point_set = set()
    points_list = []
    
    # First, generate initial random points (but track duplicates)
    initial_points = np.zeros((k, d), dtype=int)
    for dim in range(d):
        values = np.repeat(np.arange(n), repeats)
        np.random.shuffle(values)
        initial_points[:, dim] = values
    
    # Add unique points from random generation
    for point in initial_points:
        point_tuple = tuple(point)
        if point_tuple not in point_set:
            point_set.add(point_tuple)
            points_list.append(point)
    
    print(f"Added {len(points_list)} unique points from random generation. Need {k - len(points_list)} more.")
    
    # Now fill remaining points deterministically while maintaining balance
    # Track how many times each value appears in each dimension
    dim_counts = np.zeros((d, n), dtype=int)
    for point in points_list:
        for dim in range(d):
            dim_counts[dim, point[dim]] += 1
    
    # Required count for each value in each dimension
    target_count = repeats
    
    # Generate remaining points by selecting values that need to appear more
    attempts = 0
    max_fill_attempts = max_attempts * 10
    
    with tqdm(total=k - len(points_list), desc="Filling remaining points") as pbar:
        while len(points_list) < k and attempts < max_fill_attempts:
            attempts += 1
            
            # Build a new point by selecting values that are below target count
            new_point = []
            for dim in range(d):
                # Find values that still need to appear more in this dimension
                needed_values = [val for val in range(n) if dim_counts[dim, val] < target_count]
                
                if needed_values:
                    # Randomly select from needed values
                    val = np.random.choice(needed_values)
                else:
                    # All values at target, choose any (this shouldn't happen with correct k)
                    val = np.random.randint(0, n)
                
                new_point.append(val)
            
            new_point_tuple = tuple(new_point)
            
            # Check if this point is unique
            if new_point_tuple not in point_set:
                point_set.add(new_point_tuple)
                points_list.append(new_point)
                
                # Update dimension counts
                for dim in range(d):
                    dim_counts[dim, new_point[dim]] += 1
                
                pbar.update(1)
    
    # if len(points_list) < k:
    #     raise RuntimeError(f"Could not generate {k} distinct points. Only generated {len(points_list)} points.")
    
    print(f"✓ Successfully generated {k} distinct balanced points using hybrid approach.")
    # return np.array(points_list[:k])
    return np.array(points_list)

def generate_independent_set(n, k, d, max_attempts, all_points):
    if all_points:
        points = generate_all_points(n, d)
    else:
        points = generate_balanced_points(n, k, d, max_attempts)
    independent_points = create_independent_points(points)
    return independent_points

def generate_all_points(n, d):
    total_points = n ** d
    points = np.zeros((total_points, d), dtype=int)
    
    for i in range(total_points):
        for dim in range(d):
            points[i, dim] = (i // (n ** dim)) % n
    return points


def find_available_points(ind_set, other_set, d):
    available_points = []
    for point in other_set:
        is_available = True
        for ind_point in ind_set:
            # Check adjacency
            is_adjacent = True
            for dim in range(d):
                diff = abs(point[dim] - ind_point[dim])
                if diff > 1 and diff < 6:
                    is_adjacent = False
                    break
            if is_adjacent:
                is_available = False
                break
        if is_available:
            available_points.append(point)
    
    return np.array(available_points)

def extend_independent_set(ind_set, n, d):
    all_points = generate_all_points(n, d)
    other_set = np.array([point for point in all_points if point.tolist() not in ind_set.tolist()])
    available_points = find_available_points(ind_set, other_set, d)
    print(f"Found {len(available_points)} available points to extend the independent set.")
    while len(available_points) > 0:
        adjacency_counts = return_adjacencies(available_points)
        min_index = adjacency_counts.index(min(adjacency_counts))
        ind_set = np.vstack([ind_set, available_points[min_index]])
        available_points = np.delete(available_points, min_index, axis=0)
        available_points = find_available_points(ind_set, available_points, d)

    return ind_set

def extended_set(n, k, d, max_attempts, all_points):
    ind_set = generate_independent_set(n, k, d, max_attempts, all_points)
    print(f"Initial independent set generated")
    extended_set = extend_independent_set(ind_set, n, d)
    print(f"Extended independent set generated")
    return extended_set

def count_points_in_d(ind_set, d, n):
    counts = np.zeros((d, n), dtype=int)
    for point in ind_set:
        for dim in range(d):
            counts[dim, point[dim]] += 1
    return counts

def check_movements(ind_set, d, n):
    d_counts = count_points_in_d(ind_set, d, n)
    mov_numbers = np.zeros(len(ind_set))
    mov_directions = np.zeros((len(ind_set), 2), dtype=int)  # First column: dimension, second column: direction
    i = 0
    for point in ind_set:
        best_value = 0
        mov_dir = 0
        mov_value = 0
        # For every dimension, check if moving up or down is possible and whether it increases balance in the dimensions, so if the new dimension has 2 points less than the old one
        for dim in range(d):
            current_value = point[dim]
            for direction in [-1, 1]:
                new_value = (current_value + direction) % n
                d_diff = d_counts[dim, new_value] - d_counts[dim, current_value]
                if d_diff < -1:
                    #Check if new point is still independent
                    count_adj = count_adjacencies(np.array([new_value if i == dim else point[i] for i in range(d)]), ind_set)
                    if (count_adj == 1):
                        best_value = -d_diff-1
                        mov_dir = dim
                        mov_value = direction
        mov_numbers[i] = best_value
        mov_directions[i] = np.array([mov_dir, mov_value])
        i += 1
    return mov_numbers, mov_directions

def moving_independent_set(ind_set, d, n):
    mov_numbers, mov_directions = check_movements(ind_set, d, n)
    print(f"Starting moving independent set with max movement value {max(mov_numbers)}")
    count = 0
    while max(mov_numbers) > 0:
        max_index = np.argmax(mov_numbers)
        dim, direction = mov_directions[max_index]
        new_point = ind_set[max_index].copy()
        new_point[dim] = (new_point[dim] + direction) % n
        #Update independent set
        ind_set[max_index] = new_point
        #Recalculate movements
        count+=1
        mov_numbers, mov_directions = check_movements(ind_set, d, n)
    print(f"Finished moving independent set after {count} moves.")
    return ind_set 

def max_moving_independent_set(ind_set, d, n):
    max_moving = check_movements(ind_set, d, n)
    print(max_moving)
    while (max_moving[0].max() > 0):
        ind_set = moving_independent_set(ind_set, d, n)
        ind_set = extend_independent_set(ind_set, n, d)
        max_moving = check_movements(ind_set, d, n)
    return ind_set

THE FOLLOWING PIECE OF CODE RANDOMLY CREATES A EVENLY SPACED OUT SET AND THEN DELETES TILL INDEPENDENT

In [3]:
# # Example usage
# n = 7   # Number of distinct values per dimension
# k = 84  # Must be divisible by 7
# d = 3   # Number of dimensions
# max_attempts = 10000

# independent_points = generate_independent_set(n, k, d, max_attempts)
# print(f"Generated {len(independent_points)} independent points out of requested {k} points.")
# print(independent_points)

# print("Adjacency counts:", return_adjacencies(independent_points))
# lower_bound = len(independent_points)**(1/d)
# print(f"Generated set, so lower bound Shannon capacity equals: {lower_bound}")

#THE FOLLOWING PIECE OF CODE RANDOMLY CREATES EVENLY SPACED OUT SET, THEN DELETES TILL INDEPENDENT, THEN FILLS IN TILL NO OPTIONS AVAILABLE ANYMORE

In [4]:
n = 7   # Number of distinct values per dimension
k = 1400  # Must be divisible by 7
d = 5   # Number of dimensions
max_attempts = 10000
all_points = False

extended_points = extended_set(n, k, d, max_attempts, all_points)
print(f"Extended independent set size: {len(extended_points)}")
print(extended_points)
print(return_adjacencies(extended_points))
lower_bound = len(extended_points)**(1/d)
print(f"Generated set, so lower bound Shannon capacity equals: {lower_bound}")

#Check spacing between all dimsensions, so how many times every integer in every dimsenion occurs
for dim in range(d):
    values, counts = np.unique(extended_points[:, dim], return_counts=True)
    print(f"Dimension {dim}:")
    for value, count in zip(values, counts):
        print(f"  Value {value} occurs {count} times")


Random generation: 100%|██████████| 5000/5000 [00:11<00:00, 423.98it/s]


Random generation didn't succeed. Switching to hybrid approach...
Added 1344 unique points from random generation. Need 56 more.


Filling remaining points:  96%|█████████▋| 54/56 [00:08<00:00,  6.20it/s]


✓ Successfully generated 1400 distinct balanced points using hybrid approach.
Initial independent set generated
Found 169 available points to extend the independent set.
Extended independent set generated
Extended independent set size: 248
[[5 5 2 0 3]
 [1 3 5 4 5]
 [1 1 4 1 0]
 ...
 [1 1 0 6 3]
 [1 1 0 1 3]
 [2 1 0 0 1]]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [5]:
#Moving the independent set and then extending
# n = 7   # Number of distinct values per dimension
# k = 700  # Must be divisible by 7
# d = 4   # Number of dimensions
# max_attempts = 10000
# all_points = False

# extended_points = extended_set(n, k, d, max_attempts, all_points)
moved_set = max_moving_independent_set(extended_points, d, n)
print(f"Moved independent set size: {len(moved_set)}")
print(moved_set)
print(return_adjacencies(moved_set))
lower_bound = len(moved_set)**(1/d)
print(f"Generated set, so lower bound Shannon capacity equals: {lower_bound}")

#Check spacing between all dimsensions, so how many times every integer in every dimsenion occurs
for dim in range(d):
    values, counts = np.unique(moved_set[:, dim], return_counts=True)
    print(f"Dimension {dim}:")
    for value, count in zip(values, counts):
        print(f"  Value {value} occurs {count} times")


(array([ 0.,  8.,  0.,  0., 16.,  0., 15.,  0.,  8.,  0., 17., 12., 12.,
        0.,  0., 10.,  0.,  0., 17.,  4., 16.,  0.,  0.,  0., 18.,  9.,
       15.,  0.,  0.,  0.,  0., 12.,  0.,  0.,  2., 16.,  0.,  9., 17.,
        0.,  9.,  0., 20.,  0.,  0.,  0., 12.,  2., 18.,  3.,  0.,  0.,
        0., 19., 20.,  9.,  0.,  3.,  9., 17.,  0., 18.,  0., 16.,  5.,
        0.,  0.,  0.,  9.,  0., 15., 13., 19.,  0., 15.,  0., 13.,  0.,
        0.,  9.,  3.,  2.,  0.,  0.,  0.,  0.,  0.,  0.,  9.,  0.,  0.,
        0., 16.,  8., 15.,  0.,  0.,  0.,  0.,  3.,  0.,  2.,  0., 17.,
       15.,  0.,  0., 15.,  0.,  0.,  3.,  5.,  9.,  0.,  0., 21., 15.,
       19., 20., 21., 15.,  0.,  8.,  9., 12.,  0., 12.,  5.,  9.,  0.,
        0.,  0.,  8., 20.,  9., 20., 15.,  9.,  7.,  0.,  2.,  0.,  0.,
        0.,  0.,  0.,  0.,  8.,  0.,  3.,  0.,  9.,  0.,  8.,  9.,  7.,
        0.,  0.,  2.,  0.,  3.,  0., 13.,  3., 12.,  0.,  3.,  0.,  9.,
        0.,  9.,  8.,  2.,  2., 15.,  0.,  0.,  0.,  0., 12., 1